# Stage 6/7a — AfriBERTa Embedding, Near-Duplicate Detection, BERTopic, AfriSentiReads `ngx.tweets_resolved` from BigQuery, embeds text with AfriBERTa, flags near-duplicates,fits BERTopic once on the full corpus, runs AfriSenti sentiment, and writes results back toBigQuery for stage 7b (feature computation).**Run order matters.** Execute cells top to bottom. Each expensive step checkpoints to GCS,so a disconnect only costs you the current batch, not the whole run.**Before running:** confirm GPU + High-RAM below. Use L4, not T4 or A100, for this workload.

In [ ]:
from google.colab import authauth.authenticate_user()print("Authenticated.")

In [ ]:
!nvidia-smiimport psutilprint(f"{psutil.virtual_memory().total/1e9:.1f} GB RAM available")

In [ ]:
PROJECT = "ngx-discourse-2026"DATASET = "ngx"BUCKET  = "ngx-discourse-2026-raw"TWEETS_RESOLVED   = f"{DATASET}.tweets_resolved"TWEET_EMBEDDINGS  = f"{DATASET}.tweet_embeddings"TWEET_TOPICS      = f"{DATASET}.tweet_topics"TWEET_SENTIMENT   = f"{DATASET}.tweet_sentiment"TWEET_DUPLICATES  = f"{DATASET}.tweet_duplicates"EMBED_CHECKPOINT_PREFIX = f"gs://{BUCKET}/derived/embeddings/"MODEL_NAME = "castorini/afriberta_large"BATCH_SIZE = 128DUP_SIM_THRESHOLD = 0.97   # cosine similarity above this = near-duplicateprint("Config loaded.")

In [ ]:
!pip install -q "transformers==4.44.2" "torch==2.4.0" "sentence-transformers==3.0.1" \    "bertopic==0.16.3" "faiss-gpu-cu12==1.9.0" "google-cloud-bigquery==3.25.0" \    "google-cloud-storage==2.18.2" "db-dtypes==1.3.0" "pandas==2.2.2" "tqdm==4.66.5"import torch, transformers, bertopicprint("torch:", torch.__version__)print("transformers:", transformers.__version__)print("bertopic:", bertopic.__version__)print("CUDA available:", torch.cuda.is_available())if torch.cuda.is_available():    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
from google.cloud import bigqueryclient = bigquery.Client(project=PROJECT)query = f'''SELECT tweet_id, trading_day, created_at, text, text_status,       author_hash, retweet_of_id, reply_to_hashFROM `{PROJECT}.{TWEETS_RESOLVED}`WHERE in_analysis_window  AND text_status != 'truncated'  AND text IS NOT NULL  AND LENGTH(text) > 0'''df = client.query(query).result().to_dataframe()print(f"Loaded {len(df):,} tweets with usable text")print(df['text_status'].value_counts())

In [ ]:
import timeimport torchfrom transformers import AutoTokenizer, AutoModeldevice = "cuda" if torch.cuda.is_available() else "cpu"tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)model = AutoModel.from_pretrained(MODEL_NAME).to(device)model.eval()print(f"Model loaded on {device}")@torch.no_grad()def embed_batch(texts, max_length=128):    enc = tokenizer(texts, padding=True, truncation=True, max_length=max_length,                     return_tensors="pt").to(device)    out = model(**enc)    cls = out.last_hidden_state[:, 0, :]  # [CLS] token    return cls.cpu().numpy()# Timing test on a 5,000-tweet sample to extrapolate the full runsample = df.sample(min(5000, len(df)), random_state=42).reset_index(drop=True)t0 = time.time()for i in range(0, len(sample), BATCH_SIZE):    batch_texts = sample['text'].iloc[i:i+BATCH_SIZE].tolist()    _ = embed_batch(batch_texts)elapsed = time.time() - t0rate = len(sample) / elapsedeta_full = len(df) / rateprint(f"{len(sample)} tweets embedded in {elapsed:.1f}s -> {rate:.1f} tweets/sec")print(f"Estimated time for full corpus ({len(df):,} tweets): {eta_full/60:.1f} minutes")

In [ ]:
import numpy as npimport pandas as pdfrom google.cloud import storageimport io, timestorage_client = storage.Client(project=PROJECT)bucket = storage_client.bucket(BUCKET)CHECKPOINT_EVERY = 20   # batchesexisting_ids = set()# Resume support: check which tweet_ids are already embeddedblobs = list(storage_client.list_blobs(BUCKET, prefix="derived/embeddings/"))if blobs:    print(f"Found {len(blobs)} existing checkpoint file(s), checking for already-embedded IDs...")    for b in blobs:        if b.name.endswith(".parquet"):            buf = io.BytesIO(b.download_as_bytes())            existing_ids.update(pd.read_parquet(buf, columns=['tweet_id'])['tweet_id'].tolist())    print(f"{len(existing_ids):,} tweets already embedded, will be skipped")todo = df[~df['tweet_id'].isin(existing_ids)].reset_index(drop=True)print(f"{len(todo):,} tweets remaining to embed")t_start = time.time()buffer_ids, buffer_vecs = [], []checkpoint_num = len(blobs)for batch_num, i in enumerate(range(0, len(todo), BATCH_SIZE)):    batch = todo.iloc[i:i+BATCH_SIZE]    vecs = embed_batch(batch['text'].tolist())    buffer_ids.extend(batch['tweet_id'].tolist())    buffer_vecs.append(vecs)    if (batch_num + 1) % CHECKPOINT_EVERY == 0 or i + BATCH_SIZE >= len(todo):        all_vecs = np.vstack(buffer_vecs)        out_df = pd.DataFrame({            'tweet_id': buffer_ids,            'embedding': list(all_vecs)        })        checkpoint_num += 1        blob_name = f"derived/embeddings/part-{checkpoint_num:05d}.parquet"        buf = io.BytesIO()        out_df.to_parquet(buf, index=False)        buf.seek(0)        bucket.blob(blob_name).upload_from_file(buf, content_type="application/octet-stream")        elapsed = time.time() - t_start        done = i + len(batch)        rate = done / elapsed if elapsed > 0 else 0        remaining = (len(todo) - done) / rate if rate > 0 else 0        print(f"Checkpoint {checkpoint_num}: {done:,}/{len(todo):,} embedded "              f"({rate:.1f} tweets/sec, ~{remaining/60:.1f} min remaining)")        buffer_ids, buffer_vecs = [], []print("Embedding complete.")

In [ ]:
# Reload every checkpoint and consolidateblobs = list(storage_client.list_blobs(BUCKET, prefix="derived/embeddings/"))frames = []for b in blobs:    if b.name.endswith(".parquet"):        buf = io.BytesIO(b.download_as_bytes())        frames.append(pd.read_parquet(buf))embeddings_df = pd.concat(frames, ignore_index=True).drop_duplicates(subset='tweet_id')print(f"Total embeddings: {len(embeddings_df):,}")print(f"Embedding dimension: {len(embeddings_df['embedding'].iloc[0])}")# Write to BigQuery as an array<float64> columnembeddings_df['embedding'] = embeddings_df['embedding'].apply(lambda v: [float(x) for x in v])job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")load_job = client.load_table_from_dataframe(    embeddings_df, f"{PROJECT}.{TWEET_EMBEDDINGS}", job_config=job_config)load_job.result()print(f"Loaded {len(embeddings_df):,} rows to {TWEET_EMBEDDINGS}")

## Near-duplicate detectionUses a local FAISS index over the embeddings just computed. Flags tweets whose nearestneighbour (excluding themselves) exceeds the similarity threshold — catches templated spam("Cost of Financial Illiteracy", GTBank scam blocks) before it inflates f7/f10/f11.

In [ ]:
import faissvecs = np.stack(embeddings_df['embedding'].apply(np.array).values).astype('float32')faiss.normalize_L2(vecs)  # normalize so inner product = cosine similarityindex = faiss.IndexFlatIP(vecs.shape[1])index.add(vecs)k = 2  # self + nearest neighboursims, idxs = index.search(vecs, k)nearest_sim = sims[:, 1]          # column 0 is self (sim=1.0)nearest_idx = idxs[:, 1]is_duplicate = nearest_sim >= DUP_SIM_THRESHOLDprint(f"Flagged {is_duplicate.sum():,} / {len(vecs):,} tweets as near-duplicates "      f"({100*is_duplicate.sum()/len(vecs):.1f}%) at threshold {DUP_SIM_THRESHOLD}")dup_df = pd.DataFrame({    'tweet_id': embeddings_df['tweet_id'].values,    'nearest_neighbor_tweet_id': embeddings_df['tweet_id'].values[nearest_idx],    'nearest_similarity': nearest_sim,    'is_near_duplicate': is_duplicate})job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")client.load_table_from_dataframe(dup_df, f"{PROJECT}.{TWEET_DUPLICATES}",                                   job_config=job_config).result()print(f"Loaded duplicate flags to {TWEET_DUPLICATES}")

## BERTopic — fit once on the full corpus**Critical:** fit exactly once here. Every window is `.transform()`-ed into this fixedtopic space downstream (stage 7b), so topic IDs stay comparable across trading days.Near-duplicates are excluded from the fit so templated spam doesn't dominate a cluster.

In [ ]:
from bertopic import BERTopicfrom sklearn.feature_extraction.text import CountVectorizerfit_mask = ~dup_df.set_index('tweet_id').loc[embeddings_df['tweet_id']]['is_near_duplicate'].valuesfit_texts = df.set_index('tweet_id').loc[embeddings_df['tweet_id'][fit_mask]]['text'].tolist()fit_vecs = vecs[fit_mask]print(f"Fitting BERTopic on {len(fit_texts):,} de-duplicated tweets...")vectorizer_model = CountVectorizer(stop_words=None, min_df=5)topic_model = BERTopic(    embedding_model=None,   # we supply precomputed embeddings    vectorizer_model=vectorizer_model,    calculate_probabilities=False,    verbose=True)t0 = time.time()topics, _ = topic_model.fit_transform(fit_texts, embeddings=fit_vecs)print(f"BERTopic fit in {(time.time()-t0)/60:.1f} minutes, "      f"{len(set(topics)) - (1 if -1 in topics else 0)} topics found")# Persist the fitted model to GCS so it can be reloaded in stage 7b for .transform()import osos.makedirs("/content/bertopic_model", exist_ok=True)topic_model.save("/content/bertopic_model/model", serialization="pickle")bucket.blob("derived/bertopic_model/model").upload_from_filename("/content/bertopic_model/model")print("BERTopic model saved to GCS at derived/bertopic_model/model")

In [ ]:
# Transform ALL tweets (including near-duplicates) into the fixed topic spaceall_texts = df.set_index('tweet_id').loc[embeddings_df['tweet_id']]['text'].tolist()all_topics, _ = topic_model.transform(all_texts, embeddings=vecs)topics_df = pd.DataFrame({    'tweet_id': embeddings_df['tweet_id'].values,    'topic_id': all_topics})job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")client.load_table_from_dataframe(topics_df, f"{PROJECT}.{TWEET_TOPICS}",                                   job_config=job_config).result()print(f"Loaded topic assignments to {TWEET_TOPICS}")

## AfriSenti — sentiment classificationProduces per-tweet bullish/neutral/bearish probabilities feeding f12 (sentiment skew).

In [ ]:
from transformers import AutoModelForSequenceClassification, pipelineSENTIMENT_MODEL = "Davlan/afrisenti-twitter-sentiment-afriberta-large"  # verify against AfriSenti repo before final runsent_tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODEL)sent_model = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_MODEL).to(device)sent_pipe = pipeline("text-classification", model=sent_model, tokenizer=sent_tokenizer,                      device=0 if device == "cuda" else -1, top_k=None, truncation=True,                      max_length=128, batch_size=BATCH_SIZE)t0 = time.time()results = sent_pipe(df['text'].tolist())print(f"Sentiment scored in {(time.time()-t0)/60:.1f} minutes")# Flatten to a bullish-probability column (adjust label names to match model's actual output)def bullish_prob(scores):    d = {s['label'].lower(): s['score'] for s in scores}    return d.get('positive', d.get('bullish', 0.0))sent_df = pd.DataFrame({    'tweet_id': df['tweet_id'].values,    'bullish_prob': [bullish_prob(r) for r in results]})job_config = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")client.load_table_from_dataframe(sent_df, f"{PROJECT}.{TWEET_SENTIMENT}",                                   job_config=job_config).result()print(f"Loaded sentiment scores to {TWEET_SENTIMENT}")

## VerificationConfirms row counts line up before moving to stage 7b (feature computation).

In [ ]:
print("Tweets embedded:      ", len(embeddings_df))print("Duplicate flags:      ", len(dup_df), f"({dup_df['is_near_duplicate'].mean()*100:.1f}% flagged)")print("Topic assignments:    ", len(topics_df))print("Sentiment scores:     ", len(sent_df))print("Source tweets (input):", len(df))